# 03 — Local SLM Battery Control · Google Colab

**Phase 1 / Month 2** · MSc Thesis — ECLIPSE project  
Supervisor: Dr. Panagiotis Kasnesis | Student: Antonios Bastoulis

---

This notebook replicates the dual-agent experiment from `02_llm_policy.ipynb` but runs the
language model **locally on the Colab GPU** instead of making remote API calls.

| Approach | Model | Latency / call | 168-step rollout |
|----------|-------|----------------|------------------|
| `02` Remote API | claude-haiku / deepseek-chat | 0.5–3 s | **1–2 hours** |
| **`03` Local SLM** | Qwen2.5-1.5B (T4) | ~0.3 s | **~2 minutes** |
| **`03` Local SLM** | Qwen3-4B (T4) | ~0.8 s | **~5 minutes** |

**Setup:** Runtime → Change runtime type → **T4 GPU** (free tier).  
Fill in `GITHUB_REPO` in § 0, then **Run All**.

**Dual-agent setup** (same as `02_llm_policy`):

| Agent | Buildings | Observability |
|-------|-----------|---------------|
| **α** | B0, B1, B2 | Sees only its 3 buildings |
| **β** | B3, B4, B5 | Sees only its 3 buildings |

Both agents share the same model weights — inference is serial (α then β) with no crosstalk.

## § 0 — Runtime & Config
> **Edit this cell only.** Nothing else needs changing.

In [ ]:
import os, sys, subprocess, time, warnings, json, random
import numpy as np

# ── Runtime check ─────────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        _gpu  = torch.cuda.get_device_name(0)
        _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ GPU: {_gpu}  ({_vram:.1f} GB VRAM)")
    else:
        print("⚠  No GPU — set Runtime → Change runtime type → T4 GPU")
except ImportError:
    print("torch not yet installed — will be in § 1")

# ── Model selection ────────────────────────────────────────────────────────
# Presets (uncomment one):
#   "Qwen/Qwen2.5-1.5B-Instruct"      1.5B · ~3 GB  · T4 free · ~0.3 s/call  ← default
#   "Qwen/Qwen3-4B"                    4B   · ~8 GB  · T4 free · ~0.8 s/call
#   "microsoft/Phi-3.5-mini-instruct"  3.8B · ~8 GB  · T4 free · ~0.8 s/call
#   "Qwen/Qwen3-8B"                    8B   · needs 4-bit on T4 or A100
#   "meta-llama/Llama-3.2-3B-Instruct" 3B   · gated · set HF_TOKEN below
MODEL_ID:       str  = "Qwen/Qwen2.5-1.5B-Instruct"
LOAD_IN_4BIT:   bool = False   # True for ≥7B models on T4
MAX_NEW_TOKENS: int  = 400     # 400=allows reasoning; 128=fast/direct output

# ── HuggingFace token (only needed for gated models like Llama) ───────────
HF_TOKEN: str | None = None    # or os.environ.get("HF_TOKEN")

# ── GitHub repo URL ────────────────────────────────────────────────────────
# Replace with your actual GitHub URL before running!
GITHUB_REPO: str = "https://github.com/YOUR_USERNAME/eclipse-thesis.git"

# ── Experiment window (matches 02_llm_policy for direct comparison) ────────
WEEK_START: int = 3624
WEEK_LEN:   int = 168

# ── Dual-agent split ───────────────────────────────────────────────────────
AGENT_A_BUILDINGS: list = [0, 1, 2]
AGENT_B_BUILDINGS: list = [3, 4, 5]

# ── Google Drive (persist results across sessions) ─────────────────────────
MOUNT_DRIVE:  bool = False
DRIVE_OUTDIR: str  = "/content/drive/MyDrive/eclipse-thesis/results"

# ── Local output ───────────────────────────────────────────────────────────
LOCAL_OUTDIR: str = "/content/eclipse-thesis/notebooks/artifacts"
os.makedirs(LOCAL_OUTDIR, exist_ok=True)

print(f"\nModel          : {MODEL_ID}")
print(f"4-bit quant    : {LOAD_IN_4BIT}")
print(f"Max new tokens : {MAX_NEW_TOKENS}")
print(f"Window         : t{WEEK_START}..{WEEK_START + WEEK_LEN - 1} ({WEEK_LEN} steps)")
print(f"Agents         : α={AGENT_A_BUILDINGS}  β={AGENT_B_BUILDINGS}")
print(f"Mount Drive    : {MOUNT_DRIVE}")

## § 1 — Install Dependencies

In [ ]:
!pip install -q citylearn transformers accelerate bitsandbytes
print("✓ Dependencies installed")

## § 2 — Clone Repository

This pulls `src/env.py` and `src/agent.py` from your GitHub repo.  

> **Alternative if you haven't pushed yet:** upload `src/env.py` and `src/agent.py`
> via the Colab Files panel (left sidebar → upload icon) into `/content/src/`,
> then replace `REPO_DIR` below with `"/content"` and skip the `git clone` block.

In [ ]:
REPO_DIR = "/content/eclipse-thesis"

if not os.path.exists(REPO_DIR):
    result = subprocess.run(
        ["git", "clone", GITHUB_REPO, REPO_DIR],
        capture_output=True, text=True,
    )
    print(result.stdout or result.stderr)
else:
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "pull"],
        capture_output=True, text=True,
    )
    print("Repo already present —", result.stdout.strip() or "up to date")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"sys.path ← {REPO_DIR}")

# HuggingFace login (only for gated models like Llama)
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✓ HuggingFace login OK")

## § 3 — Imports & Environment Factory

`make_colab_env()` is a Colab-friendly variant of `src.env.make_env()`.  
It passes the dataset name as a string to `CityLearnEnv`, which triggers an automatic
download (~50 MB) on the first call. Subsequent calls use the cached copy.

In [ ]:
import logging
import pandas as pd
import matplotlib.pyplot as plt
import citylearn
from citylearn.citylearn import CityLearnEnv

from src.env import (
    SEED, BUILDINGS, snapshot_state,
    MERLINReward, EcoPeakBatteryReward,
    OBSERVATIONS_LLM, OBSERVATIONS_SAC, ACTIVE_ACTIONS,
)
from src.agent import (
    make_system_prompt, render_state, parse_actions, _ACTION_RE,
    price_bucket, solar_bucket, make_policy_llm, PRICE_PEAK_THRESHOLD,
)

warnings.filterwarnings("ignore")
np.random.seed(SEED)
random.seed(SEED)

print(f"CityLearn {citylearn.__version__}")
print("src.env and src.agent loaded.")


def make_colab_env(
    start: int = 0,
    end: int = 8758,
    obs_set: str = "llm",
    reward_fn: str = "merlin",
    buildings=None,
) -> CityLearnEnv:
    """Colab variant of src.env.make_env() — uses CityLearn auto-download schema.

    Passing the dataset name as a string triggers a one-time download.
    All other parameters match make_env().
    """
    observations = OBSERVATIONS_SAC if obs_set == "sac" else OBSERVATIONS_LLM
    env = CityLearnEnv(
        schema="citylearn_challenge_2022_phase_all",
        buildings=buildings if buildings is not None else BUILDINGS,
        central_agent=False,
        active_actions=ACTIVE_ACTIONS,
        active_observations=observations,
        random_seed=SEED,
        simulation_start_time_step=start,
        simulation_end_time_step=end,
    )
    env.reward_function = (
        EcoPeakBatteryReward(env.get_metadata()) if reward_fn == "eco"
        else MERLINReward(env.get_metadata())
    )
    return env


print("make_colab_env() ready.")
print("Note: first call downloads the dataset (~50 MB) — ~30 s on Colab.")

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTDIR, exist_ok=True)
    print(f"✓ Drive mounted → results also saved to: {DRIVE_OUTDIR}")
else:
    print("Drive not mounted (MOUNT_DRIVE=False).")
    print("Results saved to LOCAL_OUTDIR only — lost when the Colab session ends.")
    print("Set MOUNT_DRIVE=True in § 0 to persist results across sessions.")

## § 4 — Local SLM Provider

`LocalHFProvider` is a drop-in for `src.agent.LLMProvider` — it implements the same
`.complete()` and `.step()` interface, so `make_policy_llm()` works unchanged.

Key design choices:
- **Greedy decoding** (`do_sample=False`) — deterministic and reproducible
- **No timeout needed** — local GPU calls finish in < 2 s
- **Thinking mode disabled** for Qwen3 — saves ~100 tokens/call, 2× faster
- **Retry logic** — same as `LLMProvider.step()`: retries up to `max_retries` times
  if no `<action>` tags are found, then falls back to zeros

In [ ]:
_logger = logging.getLogger("local_slm")


class LocalHFProvider:
    """HuggingFace local model provider. Drop-in for src.agent.LLMProvider.

    Runs inference on the Colab GPU. No API keys, no rate limits, no latency.
    On a T4 with Qwen2.5-1.5B each call takes ~0.3 s → a 168-step dual-agent
    rollout completes in ~2 minutes instead of 1+ hour via remote APIs.

    Args:
        model_id:       HuggingFace model ID.
        load_in_4bit:   Use 4-bit NF4 quantization (bitsandbytes). Halves VRAM
                        at slight quality cost. Recommended for ≥7B models on T4.
        max_new_tokens: Max tokens generated per call. 400 allows a brief
                        chain-of-thought; 128 forces direct output (faster).
    """

    def __init__(
        self,
        model_id: str,
        load_in_4bit: bool = False,
        max_new_tokens: int = 400,
    ):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM

        self.model_id       = model_id
        self.name           = "local_hf"
        self.label          = f"local:{model_id.split('/')[-1]}"
        self.max_new_tokens = max_new_tokens
        self._device        = "cuda" if torch.cuda.is_available() else "cpu"
        self._is_qwen3      = "Qwen3" in model_id or "qwen3" in model_id.lower()

        print(f"Loading {model_id} on {self._device} …")
        load_kw: dict = {"device_map": "auto"}
        if load_in_4bit:
            from transformers import BitsAndBytesConfig
            load_kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
        else:
            load_kw["torch_dtype"] = (
                torch.bfloat16 if self._device == "cuda" else torch.float32
            )

        self.model = AutoModelForCausalLM.from_pretrained(model_id, **load_kw)
        self.model.eval()
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        import torch
        n_params = sum(p.numel() for p in self.model.parameters()) / 1e9
        mem_gb   = torch.cuda.memory_allocated() / 1e9 if self._device == "cuda" else 0.0
        print(f"  ✓ {n_params:.2f}B params | GPU mem: {mem_gb:.1f} GB")
        if self._is_qwen3:
            print("  Qwen3 detected — thinking mode disabled (enable_thinking=False)")

    def complete(
        self,
        system: str,
        user: str,
        max_tokens: int | None = None,
        **kwargs,
    ) -> str:
        """Generate a response. Returns newly generated text only."""
        import torch

        max_new  = max_tokens or self.max_new_tokens
        messages = [
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ]
        # Disable Qwen3 thinking mode to keep responses short and fast
        chat_kw = {"enable_thinking": False} if self._is_qwen3 else {}

        input_ids = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            **chat_kw,
        ).to(self._device)
        attention_mask = torch.ones_like(input_ids)

        with torch.no_grad():
            output_ids = self.model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
            )

        new_tokens = output_ids[0][input_ids.shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def step(
        self,
        state_text: str,
        system: str | None = None,
        n_buildings: int = 6,
        max_retries: int = 2,
        **kwargs,   # absorbs timeout_s etc. from make_policy_llm
    ) -> tuple[list[float], str, bool]:
        """Query the model for one environment step.

        Returns (actions, raw_response, used_fallback).
        """
        _system  = system or make_system_prompt(n_buildings)
        last_raw = ""

        for attempt in range(max_retries):
            last_raw = self.complete(_system, f"STATE:\n{state_text}")
            if _ACTION_RE.search(last_raw):
                return parse_actions(last_raw, n_buildings), last_raw, False
            _logger.debug(
                "No action tags (attempt %d): %s", attempt + 1, last_raw[:80]
            )

        _logger.warning("LocalHF fallback model=%s — returning zeros", self.model_id)
        return [0.0] * n_buildings, last_raw, True


print("LocalHFProvider defined.")
print(f"Will use: {MODEL_ID}  |  4-bit={LOAD_IN_4BIT}  |  max_new_tokens={MAX_NEW_TOKENS}")

## § 5 — Load Model

First run downloads the model weights from HuggingFace Hub (~3 GB for 1.5B, ~8 GB for 4B).  
Subsequent runs load from the Colab cache — much faster.

**VRAM guide (T4 = 15 GB):**

| Model | bf16 VRAM | 4-bit VRAM |
|-------|-----------|------------|
| Qwen2.5-1.5B | ~3 GB | — |
| Qwen3-4B / Phi-3.5-mini | ~8 GB | ~4 GB |
| Qwen3-8B / Llama-3.1-8B | ~16 GB ❌ | ~8 GB ✓ |

In [ ]:
_t0 = time.time()
slm = LocalHFProvider(
    model_id=MODEL_ID,
    load_in_4bit=LOAD_IN_4BIT,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f"\nModel loaded in {time.time() - _t0:.1f}s")

In [ ]:
# ── Warmup + speed benchmark ──────────────────────────────────────────────
print("Warmup inference …")
_wt0  = time.time()
_resp = slm.complete("You are a helpful assistant.", "Say READY in one word.")
_wt1  = time.time()
print(f"Response   : {_resp.strip()!r}")
print(f"Latency    : {_wt1 - _wt0:.2f}s")

_calls_per_run = WEEK_LEN * 2   # two agents per step
_est_min = (_wt1 - _wt0) * _calls_per_run / 60
print(f"\nEstimated rollout time: ~{_est_min:.1f} min  ({_calls_per_run} calls × {_wt1 - _wt0:.2f}s/call)")

## § 6 — Environment & State Check

`snapshot_state()` returns **12 fields** per building: 9 real-time signals + 3 forecasts
(price +6 h, price +12 h, solar irradiance +6 h). `render_state()` formats these into the
LLM prompt string, inserting a **Forecast** line.

In [ ]:
print("Building environment (first call downloads dataset if needed) …")
_env = make_colab_env(start=WEEK_START, end=WEEK_START + WEEK_LEN - 1)
_obs, _ = _env.reset()
print(f"\nBuildings      : {len(_obs)}")
print(f"Obs / building : {len(_obs[0])}  (llm obs set — 9 real-time vars in obs vector)")
print(f"Steps / episode: {_env.time_steps}")
print(f"Reward fn      : {type(_env.reward_function).__name__}")

_snap = snapshot_state(_env)
print(f"\nsnapshot_state() returns {len(_snap[0])} fields per building (9 real-time + 3 forecasts)")

print("\nForecast availability at t=0 (None = column missing in dataset):")
for k in ["electricity_pricing_predicted_1",
           "electricity_pricing_predicted_2",
           "solar_irradiance_predicted_1"]:
    print(f"  {k}: {_snap[0][k]}")

print("\n" + "="*60)
print("Agent α state (buildings 0-2):")
print("="*60)
print(render_state(_snap[0:3]))

print("\n" + "="*60)
print("Agent β state (buildings 3-5):")
print("="*60)
print(render_state(_snap[3:6]))

## § 7 — Reference Policies

In [ ]:
def policy_noop(snap: list[dict], t: int) -> list[float]:
    return [0.0] * len(snap)

_rng = np.random.default_rng(SEED)

def policy_random(snap: list[dict], t: int) -> list[float]:
    return _rng.uniform(-1.0, 1.0, size=len(snap)).tolist()

def policy_rbc(snap: list[dict], t: int) -> list[float]:
    """Price + solar aware rule-based controller.

    Exploits battery asymmetry: charge small (avoids demand spikes),
    discharge full (-1.0 is hardware-capped and safe).
    """
    acts = []
    for d in snap:
        soc = d["electrical_storage_soc"]
        prc = d["electricity_pricing"]
        sol = d["solar_generation"]
        if solar_bucket(sol) == "HIGH" and soc < 0.85:
            acts.append(0.2)          # absorb free solar
        elif price_bucket(prc) == "PEAK" and soc > 0.10:
            acts.append(-1.0)         # discharge during peak
        elif price_bucket(prc) == "LOW" and soc < 0.90:
            acts.append(0.25)         # trickle-charge when cheap
        else:
            acts.append(0.0)
    return acts

print("Reference policies defined: noop, random, rbc.")

## § 8 — Rollout Functions

In [ ]:
def run_policy(
    name: str,
    policy_fn,
    start: int = WEEK_START,
    length: int = WEEK_LEN,
    reward_fn: str = "merlin",
) -> tuple[pd.DataFrame, CityLearnEnv, list[dict]]:
    """Single-agent rollout — policy_fn sees all 6 buildings.

    policy_fn signature: (snap, t) -> list[float]  OR  (actions, raw, fallback).
    """
    env = make_colab_env(start=start, end=start + length - 1, reward_fn=reward_fn)
    env.reset()

    rows, raw_log = [], []
    done, t, t0 = False, 0, time.time()

    while not done:
        snap   = snapshot_state(env)
        result = policy_fn(snap, t)

        if isinstance(result, tuple):
            acts, raw, fb = result
            raw_log.append({"t": t, "state_text": render_state(snap), "raw": raw, "fallback": bool(fb)})
        else:
            acts = result

        _obs, reward, terminated, truncated, _ = env.step([[float(a)] for a in acts])
        done = bool(terminated or truncated)
        post = snapshot_state(env)

        rows.append({
            "policy": name, "t": t, "price": snap[0]["electricity_pricing"],
            "reward_sum": float(np.sum(reward)),
            **{f"a{i}":   acts[i]                                     for i in range(6)},
            **{f"r{i}":   float(reward[i])                            for i in range(6)},
            **{f"soc{i}": post[i]["electrical_storage_soc"]           for i in range(6)},
            **{f"net{i}": post[i]["net_electricity_consumption_last"] for i in range(6)},
        })
        t += 1

    df    = pd.DataFrame(rows)
    n_fb  = sum(1 for r in raw_log if r["fallback"])
    fb_msg = f" | fallbacks={n_fb}/{len(raw_log)}" if raw_log else ""
    print(f"[{name}] {t} steps in {time.time()-t0:.1f}s | reward={df['reward_sum'].sum():.4f}{fb_msg}")
    return df, env, raw_log

In [ ]:
def run_policy_dual_agent(
    name: str,
    policy_a,
    policy_b,
    agent_a_bldgs: list[int] = AGENT_A_BUILDINGS,
    agent_b_bldgs: list[int] = AGENT_B_BUILDINGS,
    start: int = WEEK_START,
    length: int = WEEK_LEN,
    reward_fn: str = "merlin",
) -> dict:
    """Dual-agent rollout: two LLM calls per step, partial observability.

    Agent α receives snap[agent_a_bldgs], Agent β receives snap[agent_b_bldgs].
    Their actions are combined in global building-index order before env.step().

    Returns dict with keys: df, env, raw_log_a, raw_log_b.
    """
    n_a = len(agent_a_bldgs)
    n_b = len(agent_b_bldgs)

    env = make_colab_env(start=start, end=start + length - 1, reward_fn=reward_fn)
    env.reset()

    rows, raw_log_a, raw_log_b = [], [], []
    done, t, t0 = False, 0, time.time()

    while not done:
        snap   = snapshot_state(env)
        snap_a = [snap[i] for i in agent_a_bldgs]
        snap_b = [snap[i] for i in agent_b_bldgs]

        result_a = policy_a(snap_a, t)
        result_b = policy_b(snap_b, t)

        if isinstance(result_a, tuple):
            acts_a, raw_a, fb_a = result_a
            raw_log_a.append({"t": t, "state_text": render_state(snap_a), "raw": raw_a, "fallback": bool(fb_a)})
        else:
            acts_a, fb_a = result_a, False

        if isinstance(result_b, tuple):
            acts_b, raw_b, fb_b = result_b
            raw_log_b.append({"t": t, "state_text": render_state(snap_b), "raw": raw_b, "fallback": bool(fb_b)})
        else:
            acts_b, fb_b = result_b, False

        # Combine α + β into a single 6-action vector in global building order
        acts_combined = [0.0] * (n_a + n_b)
        for local_i, global_i in enumerate(agent_a_bldgs):
            acts_combined[global_i] = acts_a[local_i]
        for local_i, global_i in enumerate(agent_b_bldgs):
            acts_combined[global_i] = acts_b[local_i]

        _obs, reward, terminated, truncated, _ = env.step([[float(a)] for a in acts_combined])
        done = bool(terminated or truncated)
        post = snapshot_state(env)

        rows.append({
            "policy": name, "t": t, "price": snap[0]["electricity_pricing"],
            "reward_sum": float(np.sum(reward)),
            "reward_a":   float(sum(reward[i] for i in agent_a_bldgs)),
            "reward_b":   float(sum(reward[i] for i in agent_b_bldgs)),
            "fallback_a": fb_a,
            "fallback_b": fb_b,
            **{f"a{i}":   acts_combined[i]                            for i in range(6)},
            **{f"r{i}":   float(reward[i])                            for i in range(6)},
            **{f"soc{i}": post[i]["electrical_storage_soc"]           for i in range(6)},
            **{f"net{i}": post[i]["net_electricity_consumption_last"] for i in range(6)},
        })
        t += 1

    df     = pd.DataFrame(rows)
    n_fb_a = sum(1 for r in raw_log_a if r["fallback"])
    n_fb_b = sum(1 for r in raw_log_b if r["fallback"])
    print(
        f"[{name}] {t} steps in {time.time()-t0:.1f}s | "
        f"reward={df['reward_sum'].sum():.4f} "
        f"(α={df['reward_a'].sum():.4f}  β={df['reward_b'].sum():.4f}) | "
        f"fallbacks α={n_fb_a} β={n_fb_b}"
    )
    return {"df": df, "env": env, "raw_log_a": raw_log_a, "raw_log_b": raw_log_b}

## § 9 — Reference Baselines

In [ ]:
df_noop,   env_noop,   _ = run_policy("noop",   policy_noop)
df_random, env_random, _ = run_policy("random", policy_random)
df_rbc,    env_rbc,    _ = run_policy("rbc",    policy_rbc)

## § 10 — SLM Dual-Agent Rollout

Both agents share the same model weights. Inference runs serially (α then β per timestep)
with no communication between them — this is the **Action-Only coordination baseline**
that mirrors the Phase 4 multi-agent deployment.

Progress is printed every step. The `[FALLBACK]` flag means no `<action>` tags were found
after `max_retries=2` attempts — the action defaults to 0.0 (no-op) for that building.

In [ ]:
_policy_a = make_policy_llm(slm, n_buildings=3, agent_label="α")
_policy_b = make_policy_llm(slm, n_buildings=3, agent_label="β")

print(f"=== Dual-agent: {slm.label}  |  max_new_tokens={MAX_NEW_TOKENS} ===")
_t0 = time.time()
slm_run = run_policy_dual_agent(f"slm_{MODEL_ID.split('/')[-1]}", _policy_a, _policy_b)
_elapsed = time.time() - _t0
print(f"\nWall clock : {_elapsed:.0f}s  ({_elapsed/60:.1f} min)")
print(f"Steps/min  : {WEEK_LEN / (_elapsed / 60):.1f}")

## § 11 — District-Level Results

In [ ]:
def summarize_district(df: pd.DataFrame, label: str) -> dict:
    net_cols = [f"net{i}" for i in range(6)]
    dist_net = df[net_cols].sum(axis=1)
    return {
        "policy":         label,
        "total_reward":   float(df["reward_sum"].sum()),
        "total_cost_est": float((dist_net * df["price"]).sum()),
        "peak_net_kW":    float(dist_net.max()),
        "total_net_kWh":  float(dist_net.sum()),
    }


slm_df    = slm_run["df"]
slm_label = slm_run["df"]["policy"].iloc[0]

summary = pd.DataFrame([
    summarize_district(df_noop,   "noop"),
    summarize_district(df_random, "random"),
    summarize_district(df_rbc,    "rbc"),
    summarize_district(slm_df,    f"{slm_label} (dual-agent)"),
]).set_index("policy")

print("District summary — higher total_reward is better (less negative)")
display(summary.round(4))

In [ ]:
HEADLINE_KPIS = [
    "electricity_consumption_total",
    "cost_total",
    "carbon_emissions_total",
    "daily_peak_average",
    "ramping_average",
    "daily_one_minus_load_factor_average",
]


def district_kpis(env_obj) -> pd.Series:
    df = env_obj.evaluate()
    if "level" in df.columns:
        mask = df["level"].astype(str).str.lower() == "district"
    elif "name" in df.columns:
        mask = df["name"].astype(str).str.lower() == "district"
    else:
        mask = pd.Series(True, index=df.index)
    d = df[mask]
    if d.empty:
        d = df.groupby("cost_function", as_index=False)["value"].mean()
    return d.set_index("cost_function")["value"].astype(float)


kpi_table = pd.DataFrame({
    "No Control": district_kpis(env_noop),
    "Random":     district_kpis(env_random),
    "RBC":        district_kpis(env_rbc),
    slm_label:    district_kpis(slm_run["env"]),
}).sort_index().round(4)

present = [k for k in HEADLINE_KPIS if k in kpi_table.index]
print("District KPIs — lower is better; No Control ≈ 1.000")
display(kpi_table.loc[present] if present else kpi_table)

In [ ]:
if present:
    ax = kpi_table.loc[present].plot(kind="bar", figsize=(12, 5), width=0.75)
    ax.axhline(1.0, color="k", ls="--", lw=0.9, label="No Control = 1.0")
    ax.set_ylabel("KPI (lower is better)")
    ax.set_title(
        f"District KPIs — {slm.label} vs baselines  "
        f"(t={WEEK_START}..{WEEK_START+WEEK_LEN-1}, {WEEK_LEN} steps)"
    )
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

## § 12 — Per-Agent Breakdown (α vs β)

In [ ]:
def per_agent_summary(df: pd.DataFrame, agent_name: str, bldg_indices: list) -> dict:
    net_cols = [f"net{i}" for i in bldg_indices]
    r_cols   = [f"r{i}"   for i in bldg_indices]
    soc_cols = [f"soc{i}" for i in bldg_indices]
    a_cols   = [f"a{i}"   for i in bldg_indices]
    dist_net = df[net_cols].sum(axis=1)
    return {
        "agent":         agent_name,
        "buildings":     str(bldg_indices),
        "total_reward":  float(df[r_cols].sum().sum()),
        "mean_soc_pct":  float(df[soc_cols].mean().mean() * 100),
        "peak_net_kW":   float(dist_net.max()),
        "total_net_kWh": float(dist_net.sum()),
        "mean_action":   float(df[a_cols].values.mean()),
        "std_action":    float(df[a_cols].values.std()),
    }


agent_df = pd.DataFrame([
    per_agent_summary(slm_df, f"{slm.label} / α", AGENT_A_BUILDINGS),
    per_agent_summary(slm_df, f"{slm.label} / β", AGENT_B_BUILDINGS),
]).set_index("agent")

print("Per-agent breakdown — α (B0-2) vs β (B3-5)")
display(agent_df.round(4))

# Fallback counts
n_fb_a = sum(1 for r in slm_run["raw_log_a"] if r["fallback"])
n_fb_b = sum(1 for r in slm_run["raw_log_b"] if r["fallback"])
print(f"\nFallbacks: α={n_fb_a}/{WEEK_LEN}  β={n_fb_b}/{WEEK_LEN}  (target: 0)")

## § 13 — Diagnostics

In [ ]:
# 13.1  SoC trajectories — α in blue, β in red, PEAK price shaded gold
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

colors_a = ["#1f77b4", "#aec7e8", "#6baed6"]
colors_b = ["#d62728", "#ff9896", "#e6550d"]

for ax, (df_, label) in zip(axes, [(df_rbc, "RBC (ref)"), (slm_df, slm.label)]):
    for li, gi in enumerate(AGENT_A_BUILDINGS):
        ax.plot(df_["t"], df_[f"soc{gi}"] * 100,
                label=f"α-B{gi}", lw=1.4, color=colors_a[li])
    for li, gi in enumerate(AGENT_B_BUILDINGS):
        ax.plot(df_["t"], df_[f"soc{gi}"] * 100,
                label=f"β-B{gi}", lw=1.4, color=colors_b[li], ls="--")

    # Shade PEAK price hours
    peak_mask = (df_["price"] >= PRICE_PEAK_THRESHOLD).values
    in_peak, span_s = False, 0
    for im, is_peak in enumerate(peak_mask):
        if is_peak and not in_peak:
            span_s = im; in_peak = True
        elif not is_peak and in_peak:
            ax.axvspan(span_s, im - 1, color="gold", alpha=0.12); in_peak = False
    if in_peak:
        ax.axvspan(span_s, len(peak_mask) - 1, color="gold", alpha=0.12)

    ax.set_xlabel("hour t"); ax.set_ylabel("SoC (%)")
    ax.set_title(label, fontsize=9)
    ax.legend(ncol=2, fontsize=6.5); ax.grid(alpha=0.3)

plt.suptitle(
    "SoC trajectories — blue=α (B0-2), red=β (B3-5) | gold=PEAK price",
    fontsize=9,
)
plt.tight_layout()
plt.show()

In [ ]:
# 13.2  District net load comparison
fig, ax = plt.subplots(figsize=(13, 4))

for df_, lbl, color, ls in [
    (df_noop,   "No Control", "gray",        "-"),
    (df_rbc,    "RBC",        "steelblue",   "-"),
    (slm_df,    slm.label,   "darkorange",  "-"),
]:
    net = df_[[f"net{i}" for i in range(6)]].sum(axis=1)
    ax.plot(df_["t"], net, label=lbl, lw=1.5, color=color, ls=ls, alpha=0.85)

ax.set_xlabel("hour t")
ax.set_ylabel("district net load (kWh)")
ax.set_title("District net load — SLM vs baselines")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 13.3  Sample raw responses — 2 timesteps, both agents
SHOW_STEPS = [10, 80]

log_a = slm_run["raw_log_a"]
log_b = slm_run["raw_log_b"]

for step_t in SHOW_STEPS:
    idx = next((i for i, r in enumerate(log_a) if r["t"] == step_t), None)
    if idx is None:
        continue
    print("═" * 70)
    print(f"t = {step_t}")
    print(f"\n── Agent α ──")
    print(f"State:\n{log_a[idx]['state_text']}")
    print(f"\nResponse (fallback={log_a[idx]['fallback']}):\n{log_a[idx]['raw']}")
    if idx < len(log_b):
        print(f"\n── Agent β ──")
        print(f"State:\n{log_b[idx]['state_text']}")
        print(f"\nResponse (fallback={log_b[idx]['fallback']}):\n{log_b[idx]['raw']}")
    print()

In [ ]:
# 13.4  Timing analysis — tokens generated per call, tokens/second
all_responses = (
    [r["raw"] for r in slm_run["raw_log_a"]]
    + [r["raw"] for r in slm_run["raw_log_b"]]
)
token_counts = [
    len(slm.tokenizer.encode(r, add_special_tokens=False))
    for r in all_responses if r
]

print(f"Total calls          : {len(all_responses)}")
print(f"Tokens generated/call: mean={np.mean(token_counts):.0f}  "
      f"min={min(token_counts)}  max={max(token_counts)}")
print(f"Total tokens         : {sum(token_counts):,}")

# Estimate tokens/s from warmup latency (before rollout started)
_warmup_toks = len(slm.tokenizer.encode(_resp, add_special_tokens=False))
_tok_per_s   = _warmup_toks / (_wt1 - _wt0)
print(f"Approx throughput    : {_tok_per_s:.0f} tokens/s  "
      f"({_tok_per_s * 60:.0f} tokens/min)")

# Fallback rate
all_fb = (
    [r["fallback"] for r in slm_run["raw_log_a"]]
    + [r["fallback"] for r in slm_run["raw_log_b"]]
)
print(f"Fallback rate        : {sum(all_fb)}/{len(all_fb)} "
      f"({100*sum(all_fb)/len(all_fb):.1f}%)  (target: 0%)")

## § 14 — Save Results

Results are always saved to `LOCAL_OUTDIR` (within the Colab session).  
If `MOUNT_DRIVE=True`, they are also copied to Google Drive for persistence.

In [ ]:
import shutil

stamp   = time.strftime("%Y%m%d_%H%M%S")
saved   = []

# ── Rollout CSV ───────────────────────────────────────────────────────────
df_all   = pd.concat([df_noop, df_random, df_rbc, slm_df], ignore_index=True)
csv_path = os.path.join(LOCAL_OUTDIR, f"03_rollouts_{stamp}.csv")
df_all.to_csv(csv_path, index=False)
saved.append(csv_path)

# ── KPI table ─────────────────────────────────────────────────────────────
if present:
    kpi_path = os.path.join(LOCAL_OUTDIR, f"03_kpis_{stamp}.csv")
    kpi_table.loc[present].to_csv(kpi_path)
    saved.append(kpi_path)

# ── Per-agent summary ─────────────────────────────────────────────────────
agent_path = os.path.join(LOCAL_OUTDIR, f"03_per_agent_{stamp}.csv")
agent_df.to_csv(agent_path)
saved.append(agent_path)

# ── Raw LLM logs (JSON) ───────────────────────────────────────────────────
for tag, log in [("alpha", slm_run["raw_log_a"]), ("beta", slm_run["raw_log_b"])]:
    log_path = os.path.join(LOCAL_OUTDIR, f"03_raw_{slm.name}_{tag}_{stamp}.json")
    with open(log_path, "w") as f:
        json.dump(log, f, indent=2)
    saved.append(log_path)

print("Saved to LOCAL_OUTDIR:")
for p in saved:
    print(f"  {p}")

# ── Copy to Google Drive if mounted ───────────────────────────────────────
if MOUNT_DRIVE and os.path.exists("/content/drive"):
    for p in saved:
        dest = os.path.join(DRIVE_OUTDIR, os.path.basename(p))
        shutil.copy2(p, dest)
    print(f"\nAlso copied to Drive: {DRIVE_OUTDIR}")
else:
    print("\n⚠  Drive not mounted — download files manually from the Files panel before session ends.")